# DCA vs. Baseline on Colab

Runs the residual baseline and DCA directly on the Colab GPU. Select **Runtime > Change runtime type > GPU** before starting.

In [ ]:
# Install the project runner and show the allocated GPU.
%pip install -q uv
!nvidia-smi

In [ ]:
import os
import subprocess

repo_dir = os.environ.get("NANOCHAT_NOTEBOOK_DIR", "/content/nanochat")

if not os.path.isdir(repo_dir):
    subprocess.run([
        "git", "clone", "--branch", "dca-v1",
        "https://github.com/nik3348/nanochat.git", repo_dir,
    ], check=True)
else:
    subprocess.run(["git", "-C", repo_dir, "switch", "dca-v1"], check=True)
    subprocess.run(["git", "-C", repo_dir, "pull", "origin", "dca-v1"], check=True)

os.chdir(repo_dir)
print("Working directory:", os.getcwd())

In [ ]:
# Install GPU dependencies.
!uv sync --extra gpu
!uv run python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

In [ ]:
# Prepare the tokenizer and a small pretraining dataset cache.
# Increase setup_shards for a longer, more representative training run.
!uv run python -m nanochat.dataset -n 8
!uv run python -m scripts.tok_train --max-chars=2000000000 --vocab-size=32768

In [ ]:
# Experiment configuration
DEPTH = 12
DCA_K_VALUES = [0, 4]       # 0 is the residual baseline
GRN_LR_VALUES = [0.005]     # One value, or one value per DCA_K value
NUM_ITERATIONS = 100
TARGET_PARAM_DATA_RATIO = 12
DEVICE_BATCH_SIZE = 8
TOTAL_BATCH_SIZE = -1       # -1 uses nanochat's automatic batch size
WINDOW_PATTERN = "L"        # Full context; robust for Colab SDPA
RUN_PREFIX = "colab-dca-exp"
DRY_RUN = False

In [ ]:
import sys

launch_args = [
    sys.executable, "-m", "scripts.run_dca_experiment",
    "--backend=local",
    f"--depth={DEPTH}",
    "--dca-k", *[str(value) for value in DCA_K_VALUES],
    "--grn-lrs", *[str(value) for value in GRN_LR_VALUES],
    f"--num-iterations={NUM_ITERATIONS}",
    f"--target-param-data-ratio={TARGET_PARAM_DATA_RATIO}",
    f"--device-batch-size={DEVICE_BATCH_SIZE}",
    f"--total-batch-size={TOTAL_BATCH_SIZE}",
    f"--window-pattern={WINDOW_PATTERN}",
    f"--run-prefix={RUN_PREFIX}",
]
if DRY_RUN:
    launch_args.append("--dry-run")

subprocess.run(launch_args, check=True)